# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** T4 session budget (3 distinct) exhausted after three repairable environment failures: (1) Colab py3.13/torch2.11+cu130 runtime required stack fixes; (2) numpy churn from requirements install + Drive per-file rate limits (fixed: filtered numpy<2, targeted throttled registered-only download); (3) pyvista stub lacked attribute fallback for import-time pv.PolyData annotation (fixed in build_notebook.py; notebook now statically valid, ready for resume). Dataset download cell fully passed in attempt 3. No scientific execution completed; gsplat JIT never ran inside the final notebook. Resume with resume_run_dir and a fresh session.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: GaussianPlant: Structure-aligned Gaussian Splatting for 3D Reconstruction of Plants

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-52e5cb26b52aa179b3db5eec9f4bd2d5)


# GaussianPlant: Structure-aligned Gaussian Splatting for 3D Reconstruction of Plants

**Yang Yang, Risa Shinoda, Hiroaki Santo, Fumio Okura** — [arXiv:2512.14087](https://arxiv.org/abs/2512.14087)

Reproduction of **Step 2 (structure extraction)** from the authors' release
[`Yrainy0615/GaussianPlant`](https://github.com/Yrainy0615/GaussianPlant) (commit `3c745e9c3c6b`), on **one scene
(`newplant9`)** from the authors' Google Drive dataset, followed by the authors'
branch **Chamfer-distance** evaluation against the released ground-truth branch cloud.

**Execution status / scope**

- **ADAPTED PYTHON DEMONSTRATION (要適合の実演)** — see the warning below.
- Single scene, one run; **not** a reproduction of paper-level dataset results.
- Runtime: **T4 GPU required** (the author pipeline hard-codes CUDA + gsplat).
  Expected total wall time ≈ 60–90 min (estimates; see section headers).

> ### ⚠️ Adaptation notice (EN)
> The released Drive dataset **does not contain** the pretrained *feature 3DGS* clouds
> (Step-1 output with 128-d DINOv3 semantic features) that Step 2 consumes. Producing
> them faithfully requires gated DINOv3 weights + the separate Feature-3DGS codebase
> trained ~30k iterations (the authors' issue tracker reports ~2 s/iter on an RTX 5090,
> i.e. ~16 h/scene) — infeasible in a Colab session. **Substitution used here:** a
> plain 3DGS cloud is pretrained **in this notebook** using the authors' own
> `GaussianModel`/renderer (COLMAP points3D init, L1+SSIM, densification) and padded
> with 128 zero semantic channels. The paper's semantic flow is therefore **inactive**;
> the branch/leaf init degrades to the authors' colour+geometry cues. All Step-2 code
> (structure extraction, binding, graph/MST, pruning, densification, evaluation) is the
> **authors' pinned code**, with three small documented compatibility adaptations:
> (1) removal of a stray unused `from turtle import speed` in `scene/__init__.py`
> (breaks headless runs); (2) a stub `pyvista` module on the import path (pyvista is
> only used by visualization functions not on this pipeline, but its import fails on
> Colab's IPython 7.34); (3) `numpy>=2` kept (Colab runtime) although the author README
> pins `<2` for their environment — `faiss-gpu-cu12` on this runtime is built against
> the numpy 2 ABI and all author modules were verified to import with numpy 2.
> The authors did not provide this Colab adaptation; validated behaviour is limited to
> what this notebook actually executes.
>
> ### ⚠️ 適合に関する注意（JP）
> 公開データセットには Step 2 の入力となる学習済み feature 3DGS 点群が含まれないため、
> 本ノートブック内で筆者らの `GaussianModel` を用いた通常 3DGS 前段学習で代用し、
> セマンティック特徴を 0 埋め 128 次元にパディングしています。論文のセマンティック
> フローは無効で、分岐/葉の初期化は色＋幾何の手掛かりに縮退します。Step 2 本体
> （構造抽出・評価コード）は筆者らの指定コミットのものを、3 点の小さな互換修正
> （未使用 `turtle` import の削除、未使用可視化関数のための `pyvista` スタブ、
> numpy>=2 の維持）を除き無修正で実行します。

## 0. Runtime selection / 実行環境の選択

**Select Runtime → Change runtime type → T4 GPU before running.**
This reproduction is GPU-only: the authors' `train.py` calls `torch.cuda.set_device`
and uses the gsplat CUDA rasterizer. **実行前に「ランタイムのタイプを変更」で
T4 GPU を選択してください。** The next cell fails loudly if no GPU is visible.

**Run all**: Runtime → Run all. Estimated: installs ~10 min, Drive download ~10–15 min,
3DGS pretrain (7k iters) ~15–25 min, Step-2 structure extraction (7k iters) ~20–40 min,
evaluation + plots ~5 min. These are estimates, not measured guarantees.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU runtime detected. Select Runtime > Change runtime type > T4 GPU and rerun. "
    "GPU is required: the author pipeline uses the gsplat CUDA rasterizer.")
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0))

## 1. Dependencies / 依存関係のインストール

Installs the authors' pinned `requirements.txt` **with the `numpy<2` line filtered out**,
plus `gsplat` (the authors' rendering backend), `faiss-gpu-cu12`, `ninja` and `gdown`.
**Deviation from the author README:** the README pins `numpy<2` for their conda env; on
this Colab runtime `faiss-gpu-cu12` is built against the numpy 2 ABI, and all author
modules were verified to import with numpy 2 — so we keep the Colab-preinstalled numpy
unchanged (replacing numpy underneath a live kernel corrupts the import state).
日本語: 筆者らの `requirements.txt` から `numpy<2` の行を除いてインストールし、gsplat・
faiss・gdown を入れます。本ランタイムでは faiss のため numpy 2 系を維持します
（README との差異；動作確認済み）。

In [ ]:
import subprocess, sys, time, urllib.request, numpy
print("preinstalled numpy:", numpy.__version__)
assert int(numpy.__version__.split(".")[0]) >= 2, "faiss-gpu-cu12 requires numpy>=2"
t0 = time.time()
req_url = "https://raw.githubusercontent.com/Yrainy0615/GaussianPlant/3c745e9c3c6b9947de9c9269239dbdf52da1085d/requirements.txt"
req_text = "\n".join(l for l in urllib.request.urlopen(req_url, timeout=60).read().decode().splitlines()
                     if l.strip() and not l.strip().startswith("numpy"))
open("/content/requirements_filtered.txt", "w").write(req_text)
for args in (["-r", "/content/requirements_filtered.txt"],
             ["gsplat", "faiss-gpu-cu12", "ninja", "gdown"]):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                       capture_output=True, text=True)
    print("pip", args[:1], "rc", r.returncode)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-800:]); raise RuntimeError("pip install failed")
print("deps installed in %.1f min" % ((time.time() - t0) / 60))

### 1b. Get the authors' pinned code (inside Colab) / 筆者コードの取得

Blobless clone of the pinned commit **inside Colab** (`/content`, exists only for this
session). We build the `simple-knn` CUDA extension; the Feature-3DGS submodule (Step 1)
is intentionally **not** initialised. Asserts the checked-out commit matches the pin.
日本語: 指定コミットを Colab 内に取得し（blobless clone）、`simple-knn` のみビルドします。
Step 1 用の Feature-3DGS サブモジュールは初期化しません。

In [ ]:
%%bash
set -e
cd /content
if [ ! -d GaussianPlant ]; then
  git clone --filter=blob:none https://github.com/Yrainy0615/GaussianPlant.git GaussianPlant
fi
cd GaussianPlant
git fetch --filter=blob:none origin 3c745e9c3c6b9947de9c9269239dbdf52da1085d -q
git checkout -q 3c745e9c3c6b9947de9c9269239dbdf52da1085d
echo "HEAD: $(git rev-parse HEAD)"
test "$(git rev-parse HEAD)" = "3c745e9c3c6b9947de9c9269239dbdf52da1085d" || { echo "COMMIT MISMATCH"; exit 1; }
git submodule update -q --init submodules/simple-knn
pip install --no-build-isolation -q ./submodules/simple-knn
echo "simple-knn built"

### 1c. Compatibility shims + scientific stack check / 互換修正と環境確認

**(1)** Removes the authors' stray, unused `from turtle import speed` line from
`scene/__init__.py` (it pulls Tk and crashes headless runtimes; `train.py` imports this
module). **(2)** Installs a stub `pyvista` module at `/content/shims` (prepended to
`sys.path` and `PYTHONPATH`): pyvista is imported at module level by the authors'
`utils/gs_utils.py` but only used by visualization functions not on this pipeline;
its import fails because Colab's IPython 7.34 lacks `IPython.core.guarded_eval`.
**(3)** Verifies the author import chain, then triggers gsplat's JIT CUDA build
(measured 794 s ≈ 13 min on the first run of a fresh T4 runtime) with a tiny dummy
rasterization so later cells are quiet.
日本語: 未使用の `turtle` import を削除し、未使用の pyvista をスタブで置き換え、
筆者コードの import 経路を検証した上で、gsplat の CUDA カーネルを事前ビルドします
（初回は約13分）。

In [ ]:
import os, sys, time, subprocess, torch

# numpy sanity: the process-wide numpy must match what pip has on disk
import numpy as np, importlib.metadata as im
if np.__version__ != im.version("numpy"):
    raise RuntimeError(
        f"numpy mismatch: kernel has {np.__version__}, disk has {im.version('numpy')}. "
        "Runtime > Restart session, then Run all again (installs are idempotent).")

# (1) documented 1-line author-code fix
REPO_DIR = "/content/GaussianPlant"
sys.path.insert(0, "/content/shims")          # (2) pyvista stub shadows the real pkg
sys.path.insert(1, REPO_DIR)
init_py = os.path.join(REPO_DIR, "scene", "__init__.py")
src = open(init_py).read()
if "from turtle import speed" in src:
    open(init_py, "w").write(src.replace("from turtle import speed\n", ""))
    print("applied documented fix: removed stray 'from turtle import speed'")

# (2) stub pyvista module (only unused viz functions need the real one)
os.makedirs("/content/shims", exist_ok=True)
open("/content/shims/pyvista.py", "w").write(
    '"""Stub: the real pyvista is not importable on Colab IPython 7.34\n'
    '(missing IPython.core.guarded_eval). Only unused visualization helpers\n'
    'of the author code reference it; annotations like pv.PolyData resolve\n'
    'to a permissive stub."""\n'
    'class _Stub:\n'
    '    def __init__(self, *a, **k): raise RuntimeError(\n'
    '        "pyvista stub: real pyvista not importable on this runtime")\n'
    'def __getattr__(name):\n'
    '    return _Stub\n')
os.environ["PYTHONPATH"] = "/content/shims"

# (3) author import chain + simple-knn
from simple_knn._C import distCUDA2
print("simple-knn OK", distCUDA2(torch.randn(10, 3).cuda()).shape)
import scene.gaussian_model, scene.dataset_readers, gaussian_renderer, arguments
print("author import chain OK")

# (3b) gsplat JIT build
import gsplat
print("gsplat", gsplat.__version__)
t0 = time.time()
out, _, _ = gsplat.rasterization(
    torch.randn(1000, 3, device="cuda"),
    torch.nn.functional.normalize(torch.randn(1000, 4, device="cuda"), dim=-1),
    torch.rand(1000, 3, device="cuda") * 0.01,
    torch.rand(1000, device="cuda"),
    torch.rand(1000, 3, device="cuda"),
    torch.eye(4, device="cuda")[None],
    torch.tensor([[[100., 0, 50], [0, 100., 50], [0, 0, 1]]], device="cuda"),
    100, 100)
torch.cuda.synchronize()
print("gsplat JIT build + rasterize OK in %.1f s" % (time.time() - t0), tuple(out.shape))

## 2. Dataset: scene `newplant9` / データセットの取得

Downloads **only** the files needed for this pipeline from the `newplant9` folder of the
authors' Drive [gaussianplant folder](https://drive.google.com/drive/folders/12X6jU1ZrLO_QvD8X5ACeypJwc4HZQQrV)
(scene id `1nW1g93FsqHua-k6QbqvKqRt8k9h9DZuH`): `sparse/0` (COLMAP), the registered `images/`, the
per-view plant masks in `masks/`, and the two ground-truth clouds `point_cloud_branch.ply`
/ `point_cloud_leaf.ply` (evaluation only; per the authors' issue #5,
`point_cloud_branch.ply` is the GT dense branch cloud). Files are fetched one by one
(the Drive folder has >50 files per subfolder, beyond `gdown --folder`'s limit) with
skip-if-exists resume and bounded retries; the large `*_leaves` leaf-instance folders are
skipped (not used by this reproduction). 日本語: 筆者らの Drive から newplant9 の
必要ファイルのみ（COLMAP sparse・登録画像・マスク・評価用 GT 点群）を 1 ファイルずつ
取得します（`*_leaves` フォルダは本再現では不使用のためスキップ）。

In [ ]:
import re, os, sys, time, subprocess, urllib.request

FOLDER = "1nW1g93FsqHua-k6QbqvKqRt8k9h9DZuH"
ROOT = "/content/gaussianplant_data/newplant9"

def list_folder(fid):
    html = urllib.request.urlopen(
        f"https://drive.google.com/embeddedfolderview?id={fid}", timeout=60).read().decode()
    out = []
    for m in re.finditer(
            r'<div class="flip-entry" id="entry-([\w-]+)"[^>]*>.*?'
            r'href="https://drive\.google\.com/(file/d/|drive/folders/)([\w-]+).*?'
            r'flip-entry-title">([^<]+)</div>', html, re.S):
        out.append(("folder" if m.group(2) == "drive/folders/" else "file",
                    m.group(3), m.group(4)))
    return out

def download(fid, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return 0
    for attempt in range(5):
        r = subprocess.run([sys.executable, "-m", "gdown", fid, "-O", path],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.getsize(path) > 0:
            time.sleep(0.7)   # gentle throttle for Drive rate limits
            return 1
        if os.path.exists(path):
            os.remove(path)
        time.sleep(45 * (attempt + 1))   # strong backoff: Drive throttles bursts
    raise RuntimeError(f"gdown failed for {fid} -> {path}: {(r.stdout + r.stderr)[-300:]}")

t0 = time.time(); n = [0]
def show():
    if n[0] and n[0] % 50 == 0:
        print(n[0], "files downloaded %.1f min" % ((time.time() - t0) / 60), flush=True)

def dl_dir(fid, dest, only_names=None, top_files_only=False):
    os.makedirs(dest, exist_ok=True)
    for kind, eid, name in list_folder(fid):
        if "_leaves" in name or (top_files_only and kind == "folder"):
            continue   # *_leaves GT instance folders & mask subfolders are not used
        if kind == "folder":
            dl_dir(eid, os.path.join(dest, name))
        elif only_names is None or name in only_names:
            n[0] += download(eid, os.path.join(dest, name)); show()

entries = {name: (kind, eid) for kind, eid, name in list_folder(FOLDER)}
# 1) COLMAP sparse first, then the two ground-truth clouds
sparse_id = entries["sparse"][1]
sparse0 = [eid for kind, eid, name in list_folder(sparse_id) if name == "0"][0]
dl_dir(sparse0, os.path.join(ROOT, "sparse", "0"))
for ply in ("point_cloud_branch.ply", "point_cloud_leaf.ply"):
    kind, eid = entries[ply]
    if kind == "file":
        n[0] += download(eid, os.path.join(ROOT, ply)); show()

# 2) registered image names from the authors' pinned colmap_loader (numpy only)
import importlib.util
spec = importlib.util.spec_from_file_location("colmap_loader", "/content/GaussianPlant/scene/colmap_loader.py")
cl = importlib.util.module_from_spec(spec); spec.loader.exec_module(cl)
reg = {e.name for e in cl.read_extrinsics_binary(os.path.join(ROOT, "sparse", "0", "images.bin")).values()}
print("registered images:", len(reg))

# 3) registered images + their top-level .JPG masks
img_id = entries["images"][1]; msk_id = entries["masks"][1]
dl_dir(img_id, os.path.join(ROOT, "images"), only_names=reg)
masks_top = [(eid, name) for kind, eid, name in list_folder(msk_id) if kind == "file"]
os.makedirs(os.path.join(ROOT, "masks"), exist_ok=True)
for eid, name in masks_top:
    if name in reg:
        n[0] += download(eid, os.path.join(ROOT, "masks", name)); show()

print("downloaded", n[0], "new files in %.1f min" % ((time.time() - t0) / 60))
for entry in sorted(os.listdir(ROOT)):
    p = os.path.join(ROOT, entry)
    print(("dir " if os.path.isdir(p) else "file"), entry,
          "" if os.path.isdir(p) else f"{os.path.getsize(p)//1024} KiB")
print("images:", len(os.listdir(os.path.join(ROOT, "images"))),
      "| masks:", len([f for f in os.listdir(os.path.join(ROOT, "masks")) if f.endswith(".JPG")]),
      "| sparse/0:", sorted(os.listdir(os.path.join(ROOT, "sparse", "0"))))

### 2b. Input preview: image + plant mask overlay / 入力プレビュー

Shows one captured view with the authors' plant mask overlaid in red — this mask is used
by the authors' built-in `--rm_bg` route to build the plant-only point cloud. Also
validates the GT branch cloud (point count, extent) with `plyfile`. 日本語:
撮影画像 1 枚に筆者らの植物マスク（赤）を重ねて表示します。このマスクは後段の
`--rm_bg` で植物点群のみを抽出するために使われます。GT 枝点群も読み込んで検証します。

In [ ]:
import glob, os, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from plyfile import PlyData

SCENE = "/content/gaussianplant_data/newplant9"
imgs = sorted(glob.glob(os.path.join(SCENE, "images", "*")))
masks = sorted(glob.glob(os.path.join(SCENE, "masks", "*.JPG")))
print("images:", len(imgs), "masks:", len(masks))
img = Image.open(imgs[0]); msk = Image.open(masks[0]).resize(img.size)
overlay = np.array(img.convert("RGB"))
m = np.array(msk.convert("L")) > 128
overlay[m] = (0.45 * overlay[m] + 0.55 * np.array([255, 40, 40])).astype(np.uint8)
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(overlay)
ax.set_title(f"newplant9 input: {os.path.basename(imgs[0])}\nred = authors' plant mask")
ax.axis("off"); plt.show()

gt = PlyData.read(os.path.join(SCENE, "point_cloud_branch.ply"))
v = gt["vertex"].data
pts = np.stack([v["x"], v["y"], v["z"]], 1)
print("GT point_cloud_branch.ply:", pts.shape[0], "points | extent:",
      np.round(pts.min(0), 3), np.round(pts.max(0), 3))

### 2c. Parse the COLMAP scene with the authors' loader / COLMAP シーンの解析

Uses the authors' `readColmapSceneInfo` (pinned code) with `features=""` and
`depths=""` — the semantic feature maps and depths the loader would normally read are
not part of the released scene, so `CameraInfo.semantic_feature` is `None` (handled
gracefully by the pinned loader). 日本語: 筆者らのローダで COLMAP シーンを読み込みます。
セマンティック特徴マップとデプスは提供されていないため `None` として扱われます。

In [ ]:
from scene.dataset_readers import readColmapSceneInfo
scene_info = readColmapSceneInfo(SCENE, images="images", depths="", masks="masks",
                                 features="", eval=False, train_test_exp=False)
print("cameras:", len(scene_info.train_cameras), "| sparse points:",
      scene_info.point_cloud.points.shape[0], "| semantic_feature_dim:",
      scene_info.semantic_feature_dim, "| extent radius: %.3f" % scene_info.nerf_normalization["radius"])

## 3. Substituted Step-1: plain 3DGS pretraining / 前段 3DGS 学習（代用）

**Adapted stage.** Trains a plain 3D Gaussian Splatting cloud for 7,000 iterations with
the authors' own `GaussianModel`, `create_from_pcd` (COLMAP points3D + RGB, simple-knn
scales) and gsplat renderer; loss = 0.8·L1 + 0.2·(1−SSIM) (the authors' loss form),
with standard clone/split densification (adaptive gradient quantile, as the authors do
for gsplat gradients) and the standard periodic opacity reset. This replaces the paper's
DINOv3 feature-3DGS pretraining (infeasible here — see the top warning). Images are
used at resolution factor 4 (~1008 px). 日本語: 筆者らの `GaussianModel` をそのまま使い、
通常の 3DGS を 7,000 反復学習して Step 2 の入力点群の代用とします（論文の
DINOv3 feature-3DGS の代用）。

In [ ]:
import random, time
import torch
from argparse import Namespace
from scene.gaussian_model import GaussianModel
from utils.camera_utils import cameraList_from_camInfos
from gaussian_renderer import render
from utils.loss_utils import l1_loss, ssim
from arguments import OptimizationParams

DEVICE = torch.device("cuda")
opt = OptimizationParams(); opt.iterations = 7000
pipe = Namespace(convert_SHs_python=False, compute_cov3D_python=False)
cam_args = Namespace(resolution=4, data_device="cpu", train_test_exp=False)
cameras = cameraList_from_camInfos(scene_info.train_cameras, 1.0, cam_args, False, False)
extent = scene_info.nerf_normalization["radius"]
print("cameras:", len(cameras), "| render res:", cameras[0].image_width, "x", cameras[0].image_height)

gaussians = GaussianModel(3, DEVICE, "/content/pretrain")
gaussians.create_from_pcd(scene_info.point_cloud, extent, 128, speedup=True)

bg = torch.zeros(3, device=DEVICE)
viewpoint_stack = cameras.copy()
t0 = time.time()
for iteration in range(1, opt.iterations + 1):
    gaussians.update_learning_rate(iteration)
    if not viewpoint_stack:
        viewpoint_stack = cameras.copy()
    cam = viewpoint_stack.pop(random.randint(0, len(viewpoint_stack) - 1))
    pkg = render(cam, gaussians, pipe, bg)
    image = pkg["render"]
    gt_image = cam.original_image.to(DEVICE)
    Ll1 = l1_loss(image, gt_image)
    loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim(image, gt_image))
    loss.backward()
    with torch.no_grad():
        if iteration < opt.densify_until_iter:
            vis = pkg["visibility_filter"]
            gaussians.max_radii2D[vis] = torch.max(gaussians.max_radii2D[vis], pkg["radii"][vis])
            gaussians.add_densification_stats(pkg["viewspace_points"], vis)
            if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
                gg = (gaussians.xyz_gradient_accum / gaussians.denom.clamp_min(1)).norm(dim=-1)
                thr = torch.quantile(gg[gg > 0], 0.9).item() if (gg > 0).any() else 1e9
                gaussians.densify_and_prune(thr, 0.005, extent, 400, pkg["radii"])
            if iteration % opt.opacity_reset_interval == 0:
                gaussians.reset_opacity()
        if iteration < opt.iterations:
            gaussians.optimizer.step(); gaussians.optimizer.zero_grad(set_to_none=True)
        if iteration % 500 == 0 or iteration == 1:
            print(f"iter {iteration:5d} | loss {loss.item():.4f} | pts {gaussians.get_xyz.shape[0]} "
                  f"| {iteration / (time.time() - t0):.1f} it/s", flush=True)
print("pretrain done in %.1f min" % ((time.time() - t0) / 60))

### 3b. Export the substitute pretrain cloud / 代用点群の書き出し

Writes the pretrained cloud with the authors' `save_ply` (it serialises 128 semantic
channels; ours are zeros) to the exact Step-2 layout the pinned loader expects:
`newplant9/feature_pretrain/point_cloud/iteration_30000/point_cloud.ply`, and verifies
the produced PLY header. 日本語: 筆者らの `save_ply` で、Step 2 が期待する配置
`feature_pretrain/point_cloud/iteration_30000/point_cloud.ply` に保存し、
PLY ヘッダを検証します（セマンティック 128 チャンネルは 0）。

In [ ]:
import os
from plyfile import PlyData
PRETRAIN_PLY = os.path.join(SCENE, "feature_pretrain", "point_cloud", "iteration_30000", "point_cloud.ply")
os.makedirs(os.path.dirname(PRETRAIN_PLY), exist_ok=True)
gaussians.save_ply(PRETRAIN_PLY)
names = PlyData.read(PRETRAIN_PLY)["vertex"].data.dtype.names
print("saved:", PRETRAIN_PLY, "|", os.path.getsize(PRETRAIN_PLY) // 2**20, "MiB")
print("semantic channels:", sum(n.startswith("semantic_") for n in names),
      "| f_rest:", sum(n.startswith("f_rest") for n in names),
      "| label:", "label_0" in names)

### 3c. Preview the pretrained cloud / 点群プレビュー

Colour scatter of the pretrained cloud from two static views (PNG output kept as durable
evidence; no interactive widget). 日本語: 学習した点群を 2 視点からカラー散布図で
表示します。

In [ ]:
import matplotlib.pyplot as plt
xyz = gaussians.get_xyz.detach().cpu().numpy()
col = (0.2820948 * gaussians.get_features[:, 0].detach().cpu().numpy() + 0.5).clip(0, 1)
fig = plt.figure(figsize=(12, 5))
for i, (el, az) in enumerate([(20, -60), (25, 120)]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.scatter(xyz[::4, 0], xyz[::4, 1], xyz[::4, 2], s=0.3, c=col[::4])
    ax.view_init(el, az); ax.set_title(f"substitute pretrain cloud (view {i+1})")
plt.tight_layout(); plt.show()

## 4. Step 2 — authors' structure extraction, unmodified / Step 2（筆者コードを無修正で実行）

Runs the authors' `train.py` with the flags documented in the pinned README's Step-2
section (auto branch fraction, cluster_size 40, binding + axis + graph regularisers,
isolation pruning, branch densification, 7,000 iterations), plus the documented
`--load_iteration 30000 --rm_bg` route so the plant-only cloud is built from the
released 2D masks on the fly. `-r 4` matches the pretraining resolution;
`--data_device cpu` keeps the ~190 images in host RAM. `PYTHONPATH` points at the
documented pyvista stub. Full log at `/content/step2_train.log`; the cell shows the tail.
日本語: 筆者らの README 記載のコマンド（パラメータは同様）を `--rm_bg` 経路で実行します。

In [ ]:
%%bash
set -o pipefail
cd /content/GaussianPlant
export CUDA_HOME=/usr/local/cuda
export PYTHONPATH=/content/shims
python train.py \
  --source_path /content/gaussianplant_data/newplant9 \
  --root_path   /content/gaussianplant_data \
  --model_path  /content/output/newplant9/run \
  --label_init joint --branch_frac -1 --cluster_size 40 \
  --reg_bind \
  --reg_axis  --lambda_axis 1.0 \
  --reg_graph --graph_from 1000 --graph_interval 50 \
  --prune_isolated --prune_from 1500 --prune_interval 500 --prune_until 4000 \
  --densify_branch --branch_split_ratio 1.5 --max_strpr_num 4000 \
  --label_lr 0 --iterations 7000 --disable_viewer \
  --load_iteration 30000 --rm_bg \
  -r 4 --data_device cpu 2>&1 | tee /content/step2_train.log | grep -v "Training progress" | tail -45

## 5. Evaluation: branch Chamfer distance / 評価

Runs the authors' `eval_chamfer.py` unmodified: symmetric Chamfer distance (L2 and
squared, pytorch3d convention) between the recovered branch points (`branch.ply`,
bounded AppGS bound to branch StrPrs) and the released GT dense branch cloud
`point_cloud_branch.ply`. Coordinates are in metres (the paper reports mm — compare
magnitudes carefully). 日本語: 筆者らの評価スクリプトで、復元した枝点群と GT 枝点群の
Chamfer 距離を計測します（座標はメートル単位）。

In [ ]:
%%bash
set -o pipefail
cd /content/GaussianPlant
python eval_chamfer.py \
  --gt /content/gaussianplant_data/newplant9/point_cloud_branch.ply \
  --points /content/output/newplant9/run/point_cloud/iteration_7000/branch.ply \
  --cylinder /content/output/newplant9/run/point_cloud/iteration_7000/strpr_branch.ply

### 5b. GT vs prediction visual comparison / GT と予測の可視化

Static 3D scatter (PNG) comparing the released GT branch cloud (grey — the authors'
reference annotation) against this run's predicted branch points (red) and the extracted
MST skeleton (blue). Left: reference only; right: overlay. 日本語: GT の枝点群（グレー）と
本実行の予測枝点群（赤）、抽出された MST スケルトン（青）を重ねて表示します。

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from plyfile import PlyData

def load_xyz(p):
    v = PlyData.read(p)["vertex"].data
    return np.stack([v["x"], v["y"], v["z"]], 1)

gt = load_xyz(os.path.join(SCENE, "point_cloud_branch.ply"))
pred = load_xyz("/content/output/newplant9/run/point_cloud/iteration_7000/branch.ply")
try:
    mst = load_xyz("/content/output/newplant9/run/point_cloud/iteration_7000/mst.ply")
except Exception:
    mst = None
fig = plt.figure(figsize=(13, 5.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.scatter(gt[::2, 0], gt[::2, 1], gt[::2, 2], s=0.5, c="grey", label="GT branch cloud (authors)")
ax.view_init(20, -60); ax.set_title("Reference: GT branch cloud"); ax.legend()
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(gt[::2, 0], gt[::2, 1], gt[::2, 2], s=0.5, c="grey", label="GT (authors)")
ax2.scatter(pred[:, 0], pred[:, 1], pred[:, 2], s=1.0, c="red", label="Predicted branch (this run)")
if mst is not None:
    ax2.scatter(mst[:, 0], mst[:, 1], mst[:, 2], s=2.0, c="blue", label="MST skeleton (this run)")
ax2.view_init(20, -60); ax2.set_title("GT vs prediction (newplant9, one run)"); ax2.legend()
plt.tight_layout(); plt.show()
print("n_gt:", len(gt), "n_pred:", len(pred))

### 5c. Results export / 結果の表出力

Recomputes the Chamfer metrics with the authors' own function and exports one CSV row
with the run's key numbers (inside Colab). 日本語: 筆者らの chamfer 関数で再計算し、
主要な結果数値を CSV として Colab 内に保存します。

In [ ]:
import csv, json, sys
sys.path.insert(0, "/content/shims"); sys.path.insert(1, "/content/GaussianPlant")
from eval_chamfer import chamfer
res = chamfer(pred.astype("float32"), gt.astype("float32"), device="cuda")
row = {"scene": "newplant9", "commit": "3c745e9c3c6b9947de9c9269239dbdf52da1085d",
       "n_gt_branch_points": int(len(gt)), "n_pred_branch_points": int(len(pred))}
row.update({k: round(float(v), 8) for k, v in res.items()})
with open("/content/gaussianplant_newplant9_results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerow(row)
print(json.dumps(row, indent=2))

## 6. Interpretation and limitations / 解釈と限界

- The numbers above come from **one scene, one run** of the authors' Step-2 code — they
  do **not** validate the paper's dataset-level results.
- **Semantic flow inactive** (zero-padded features): the branch/leaf label init relies
  on the authors' colour (strongest cue, AUC ≈ 0.89 per the code comments) + geometry
  cues only. With real DINOv3 features the semantic term would add complementary
  evidence.
- The substituted pretrain cloud was trained 7k iterations (the paper's Step-1b uses
  30k); cloud quality directly bounds StrPr quality.
- The paper reports Chamfer in mm for its benchmark; released scenes are in metres
  (repo issue #5 discusses a similar unit question without an author reply).

**References**
- Paper: arXiv:2512.14087 (https://arxiv.org/abs/2512.14087)
- Code: https://github.com/Yrainy0615/GaussianPlant @ `3c745e9c3c6b9947de9c9269239dbdf52da1085d`
- Data: authors' Drive folder `12X6jU1ZrLO_QvD8X5ACeypJwc4HZQQrV` (scene `newplant9`)
- PhenoPaper study flow used as prior research guidance only (AI-generated,
  reviewStatus: unverified).

日本語: 本結果は 1 シーン 1 実行のものであり、論文のデータセット全体の結果を検証する
ものではありません。セマンティックフローは無効、前段点群は 7,000 反復の代用です。
数値比較の際は単位（メートル vs ミリメートル）に注意してください。